# Phase 3: Real-time Simulation & Deployment
## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## 2. Load Models and Scaler

In [2]:
try:
    model = joblib.load('../models/random_forest.pkl')
    scaler = joblib.load('../models/scaler.pkl')
    le = joblib.load('../models/label_encoder.pkl')
    selected_features = joblib.load('../models/selected_features.pkl')
    print("Model, Scaler, and Features loaded successfully!")
except Exception as e:
    print(f"Error loading models: {e}. Please ensure Notebook 02 has been fully run.")

Model, Scaler, and Features loaded successfully!


## 3. Real-time Prediction Function

In [3]:
def predict_flow(flow_dict):
    """
    Simulates real-time prediction for a single network flow.
    """
    # Convert input to dataframe to ensure proper 2D shape and column order
    df_flow = pd.DataFrame([flow_dict])
    
    # Keep only selected features
    # If a feature is missing in flow_dict, fill with 0 (or some default value)
    for f in selected_features:
        if f not in df_flow.columns:
            df_flow[f] = 0.0
            
    X_input = df_flow[selected_features]
    
    # Scale
    X_scaled = scaler.transform(X_input)
    
    # Predict
    pred_encoded = model.predict(X_scaled)[0]
    pred_label = le.inverse_transform([pred_encoded])[0]
    
    # Alert if not BENIGN
    if pred_label != 'BENIGN':
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        alert_msg = f"[{timestamp}] [ALERT] Suspicious traffic detected: {pred_label}\n"
        # Print warning (red text in terminal/notebook)
        print(f"\033[91m{alert_msg.strip()}\033[0m")
        
        # Log to file
        with open('../alerts.log', 'a') as f:
            f.write(alert_msg)
    else:
        # Optional: Print normal traffic (green text)
        print(f"\033[92mNormal traffic (BENIGN)\033[0m")
        
    return pred_label

## 4. Test the Simulation

In [5]:
# Thử nghiệm mô phỏng với 20 mẫu Benign và 20 mẫu Attack ngẫu nhiên
try:
    # 1. Load dữ liệu sạch đã lưu từ Notebook 01
    df_all = pd.read_parquet('../datasets/cleaned_data.parquet')
    
    # 2. Lấy mẫu riêng biệt để đảm bảo có cả tấn công và bình thường (để xem Alert)
    df_benign = df_all[df_all['Label'] == 'BENIGN'].sample(20)
    df_attacks = df_all[df_all['Label'] != 'BENIGN'].sample(20)
    
    # 3. Trộn ngẫu nhiên lại
    df_sample = pd.concat([df_benign, df_attacks]).sample(frac=1) 
    
    print(f"Bắt đầu mô phỏng kiểm tra trên {len(df_sample)} luồng mạng...\n")
    print("-" * 50)
    
    for _, row in df_sample.iterrows():
        # Tách dữ liệu flow (loại bỏ nhãn thực tế để model tự đoán)
        flow_data = row.drop('Label').to_dict()
        true_label = row['Label']
        
        print(f"\n[Dữ liệu thực tế là: {true_label}]")
        
        # Gọi hàm dự đoán
        predict_flow(flow_data)
        
        # Giả lập độ trễ mạng để dễ theo dõi kết quả nhảy ra
        time.sleep(0.3) 
        
    print("\n" + "-" * 50)
    print("Mô phỏng hoàn tất. Kiểm tra file 'alerts.log' để xem lịch sử cảnh báo.")
        
except Exception as e:
    print(f"Lỗi khi chạy mô phỏng: {e}")


Bắt đầu mô phỏng kiểm tra trên 40 luồng mạng...

--------------------------------------------------

[Dữ liệu thực tế là: BENIGN]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: DDoS]
[2026-05-03 10:07:38] [ALERT] Suspicious traffic detected: DDoS

[Dữ liệu thực tế là: BENIGN]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: DoS Hulk]
[2026-05-03 10:07:39] [ALERT] Suspicious traffic detected: DoS Hulk

[Dữ liệu thực tế là: BENIGN]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: PortScan]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: DoS Hulk]
[2026-05-03 10:07:40] [ALERT] Suspicious traffic detected: DoS Hulk

[Dữ liệu thực tế là: BENIGN]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: BENIGN]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: BENIGN]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: PortScan]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: PortScan]
Normal traffic (BENIGN)

[Dữ liệu thực tế là: DDoS]
[2026-05-03 10:07:42] [ALERT] Suspicious traffic detected: DDoS

[Dữ liệu thực tế là: BE